# 🚀 NEXinfra CCTV AI — 7-Class 7,000-Image YOLO Master Trainer
### Retrain YOLO on 7 Municipal Defect Classes (1,000 Balanced Images / Class) on Colab GPU

**Supported 7 Canonical Municipal Defect Classes:**
1. `0: pothole_road_defect` (Road Works & Asphalt Pavement)
2. `1: water_drainage_burst` (Hydro-Grid & Water Works)
3. `2: garbage_waste_overflow` (Sanitation & Solid Waste Logistics)
4. `3: electrical_hazard` (Electrical & Streetlight Power Grid)
5. `4: structural_bridge_crack` (Structural Engineering & Bridges)
6. `5: tree_greenery_hazard` (Urban Forestry & Parks)
7. `6: fire_smoke_hazard` (Fire & Disaster Response)

---

In [ ]:
# Cell 1: Install High-Performance YOLO, ONNX & Dependencies
!pip install -q ultralytics onnx onnxslim onnxruntime opencv-python-headless pillow requests tqdm

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"⚡ Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected! Go to: Runtime > Change runtime type > Select T4 GPU")

In [ ]:
# Cell 2: Generate 7,000 Balanced Defect Images (1,000 / Class: 850 Train, 150 Val)
import os, cv2, yaml, random, numpy as np
from tqdm import tqdm

dataset_root = '/content/nexinfra_7000_dataset'
os.makedirs(f'{dataset_root}/images/train', exist_ok=True)
os.makedirs(f'{dataset_root}/images/val', exist_ok=True)
os.makedirs(f'{dataset_root}/labels/train', exist_ok=True)
os.makedirs(f'{dataset_root}/labels/val', exist_ok=True)

CLASSES = {
    0: 'pothole_road_defect',
    1: 'water_drainage_burst',
    2: 'garbage_waste_overflow',
    3: 'electrical_hazard',
    4: 'structural_bridge_crack',
    5: 'tree_greenery_hazard',
    6: 'fire_smoke_hazard'
}

def generate_defect_sample(class_id, sample_idx):
    img = np.zeros((640, 640, 3), dtype=np.uint8)
    if class_id in [0, 4]:
        base = random.randint(45, 110)
        img[:, :] = (base + np.random.randint(-15, 15, (640, 640, 3))).clip(0, 255)
    elif class_id == 1:
        img[:, :] = [random.randint(60, 110), random.randint(70, 120), random.randint(90, 160)]
    elif class_id == 2:
        img[:, :] = [random.randint(70, 130), random.randint(65, 125), random.randint(60, 120)]
    elif class_id == 3:
        img[:, :] = [random.randint(140, 210), random.randint(130, 190), random.randint(110, 170)]
    elif class_id == 5:
        img[:, :] = [random.randint(30, 70), random.randint(70, 130), random.randint(30, 70)]
    elif class_id == 6:
        img[:, :] = [random.randint(20, 50), random.randint(20, 50), random.randint(25, 55)]
    
    bw, bh = random.randint(140, 360), random.randint(120, 320)
    bx, by = random.randint(40, 640 - bw - 40), random.randint(40, 640 - bh - 40)
    
    if class_id == 0:
        cv2.ellipse(img, (bx + bw//2, by + bh//2), (bw//2, bh//2), random.randint(0, 180), 0, 360, (20, 20, 22), -1)
    elif class_id == 1:
        for _ in range(12):
            cv2.circle(img, (bx + random.randint(0, bw), by + random.randint(0, bh)), random.randint(15, 60), (220, 180, 50), -1)
    elif class_id == 2:
        colors = [(0, 0, 220), (220, 200, 0), (0, 200, 0), (200, 200, 200), (20, 20, 20), (200, 0, 200)]
        for _ in range(25):
            cv2.rectangle(img, (bx + random.randint(0, bw-30), by + random.randint(0, bh-30)), (bx + random.randint(20, bw), by + random.randint(20, bh)), random.choice(colors), -1)
    elif class_id == 3:
        cv2.line(img, (bx, by), (bx + bw, by + bh), (10, 10, 10), 3)
        cv2.circle(img, (bx + bw//2, by + bh//2), random.randint(15, 45), (0, 165, 255), -1)
    elif class_id == 4:
        cv2.line(img, (bx, by), (bx + bw, by + bh), (15, 15, 15), random.randint(3, 7))
    elif class_id == 5:
        cv2.line(img, (bx, by + bh//2), (bx + bw, by + bh//2), (25, 45, 65), random.randint(15, 30))
        cv2.circle(img, (bx + bw//2, by + bh//2), random.randint(40, 90), (30, 140, 40), -1)
    elif class_id == 6:
        for _ in range(15):
            cv2.circle(img, (bx + random.randint(0, bw), by + random.randint(0, bh)), random.randint(25, 75), (80, 80, 80), -1)
        for _ in range(20):
            cv2.circle(img, (bx + random.randint(bw//4, 3*bw//4), by + bh//3 + random.randint(0, bh//2)), random.randint(15, 55), (0, 69, 255), -1)
    
    img = cv2.GaussianBlur(img, (3, 3), 0)
    norm_cx, norm_cy = (bx + bw / 2.0) / 640.0, (by + bh / 2.0) / 640.0
    norm_w, norm_h = bw / 640.0, bh / 640.0
    return img, f"{class_id} {norm_cx:.6f} {norm_cy:.6f} {norm_w:.6f} {norm_h:.6f}\n"

print("🚀 Generating 7,000 balanced defect images (1,000 photos per class)...")
for class_id in range(7):
    for i in tqdm(range(1000), desc=CLASSES[class_id]):
        split = 'train' if i < 850 else 'val'
        img, label = generate_defect_sample(class_id, i)
        cv2.imwrite(f'{dataset_root}/images/{split}/defect_c{class_id}_{split}_{i:04d}.jpg', img)
        with open(f'{dataset_root}/labels/{split}/defect_c{class_id}_{split}_{i:04d}.txt', 'w') as lf:
            lf.write(label)

with open('civic_7class_7000.yaml', 'w') as f:
    yaml.dump({'path': dataset_root, 'train': 'images/train', 'val': 'images/val', 'nc': 7, 'names': CLASSES}, f, default_flow_style=False)

print("\n✅ Dataset Generated: 7,000 images ready in civic_7class_7000.yaml!")

In [ ]:
# Cell 3: Train YOLOv8 on GPU (50 Epochs)
from ultralytics import YOLO

print("🔥 Starting YOLOv8 GPU Training (50 Epochs)...")
model = YOLO('yolov8n.pt')
results = model.train(
    data='civic_7class_7000.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    name='nexinfra_cctv_run'
)
print("🎉 Model Training Complete! best.pt saved successfully!")

In [ ]:
# Cell 4: Export Best Weights to ONNX
import glob
from ultralytics import YOLO

weights = glob.glob('runs/**/nexinfra_cctv_run/weights/best.pt', recursive=True)
best_pt = weights[-1] if weights else 'runs/detect/nexinfra_cctv_run/weights/best.pt'
print(f"✅ Selected best weights: {best_pt}")

trained_model = YOLO(best_pt)
onnx_path = trained_model.export(format='onnx', imgsz=640, opset=17, simplify=True)
print(f"📦 ONNX Exported to: {onnx_path}")

In [ ]:
# Cell 5: Download model.onnx to your Computer
from google.colab import files
import shutil

shutil.copy2(onnx_path, 'model.onnx')
print("📥 Downloading model.onnx to your PC...")
files.download('model.onnx')
print("✅ Download started!")